# Regression Models

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8')

import seaborn as sns

In [ ]:
diamonds = sns.load_dataset('diamonds')
diamonds.head()

In [ ]:
diamonds.info()

In [ ]:
diamonds.describe(include='all')

In our exploratory data analysis (EDA), we’ve seen some surprising relationships between the quality of diamonds and their price: low quality diamonds (poor cuts, bad colours, and inferior clarity) have higher prices.

In [ ]:
# Fair cuts have higher price?
fig, ax = plt.subplots()
sns.boxplot(data=diamonds, x='cut', y='price', ax=ax)
ax.set_xlabel("Cut")
ax.set_ylabel("Price")
ax.set_title("Diamond price by cut")
plt.show()

In [ ]:
# The worst diamond color is J (slightly yellow)
fig, ax = plt.subplots()
sns.boxplot(data=diamonds, x='color', y='price', ax=ax)
ax.set_xlabel("Color")
ax.set_ylabel("Price")
ax.set_title("Diamond price by color")
plt.show()

In [ ]:
# The worst clarity is I1 (inclusions visible to the naked eye).
fig, ax = plt.subplots()
sns.boxplot(data=diamonds, x='clarity', y='price', ax=ax)
ax.set_xlabel("Clarity")
ax.set_ylabel("Price")
ax.set_title("Diamond price by clarity")
plt.show()

Do these charts mean lower quality diamonds have higher prices? If that's the case, why do people pay higher prices for lower quality? 

Do not forget there is an important confounding variable: the weight (carat) of the diamond. The weight of the diamond is the single most important factor for determining the price of the diamond, and lower quality diamonds tend to be larger.

In [ ]:
fig, ax = plt.subplots()
ax.scatter(diamonds['carat'], diamonds['price'], alpha=0.2)
ax.set_xlabel("Carat")
ax.set_ylabel("Price")
ax.set_title("Diamond price by carat")
plt.show()

## Build a Simple Regression Model for Diamond Price

We build a simple regression model to predict diamond price by carat. 

In [ ]:
# Use sklearn for linear regression
from sklearn.linear_model import LinearRegression

In [ ]:
diamonds_y = diamonds['price']
diamonds_y

In [ ]:
# X as a dataframe (numpy array)
diamonds_X = diamonds[['carat']]
diamonds_X

In [ ]:
lm1 = LinearRegression()
lm1.fit(diamonds_X, diamonds_y)

In [ ]:
print(lm1.intercept_, lm1.coef_)

In [ ]:
# R^2 of the model
lm1.score(diamonds_X, diamonds_y)

In [ ]:
# Calculate the predicted price
diamonds_y_pred = lm1.predict(diamonds_X)

In [ ]:
carat_grid = np.linspace(diamonds['carat'].min(), diamonds['carat'].max(), 200).reshape(-1, 1)
price_grid_pred = lm1.predict(pd.DataFrame(carat_grid, columns=['carat']))

fig, ax = plt.subplots()
ax.scatter(diamonds['carat'], diamonds['price'], alpha=0.2, label="Actual data")
ax.plot(carat_grid, price_grid_pred, color="red", label="Fitted model")
ax.set_xlabel("Carat")
ax.set_ylabel("Price")
ax.set_title("Simple linear model for diamond price")
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots()
sns.scatterplot(data=diamonds, x='carat', y='price', alpha=0.2, label="Actual data", ax=ax)
ax.plot(carat_grid, price_grid_pred, color="red", label="Fitted model")
ax.set_xlabel("Carat")
ax.set_ylabel("Price")
ax.set_title("Simple linear model for diamond price")
ax.legend()
plt.show()

In [ ]:
# RMSE: This indicates how off the predicted values are from the actual values. 
from sklearn.metrics import root_mean_squared_error
lm1_rmse = root_mean_squared_error(diamonds_y, diamonds_y_pred)
lm1_rmse

## Improving the model

Next, let's make a few tweaks to our model: 
- Remove the outliers (big diamonds)
- Include other variables
- Examine non-linearity 

### Remove outliers 

In [ ]:
# How many diamonds are larger than 2.5 carats (99.7% of the data)
bigrock = diamonds.carat>2.5
bigrock.value_counts(normalize=True)

In [ ]:
# Focus on diamonds smaller than 2.5 carats. 
df = diamonds[diamonds.carat<=2.5]
df.info()

In [ ]:
df = df.reset_index().drop(columns=['index'])
df

In [ ]:
df_X = df[['carat']]
df_y = df['price']

In [ ]:
# Build a new linear model using data without the outliers 
lm2 = LinearRegression()
lm2.fit(df_X, df_y)

In [ ]:
print(lm2.intercept_, lm2.coef_)

In [ ]:
# R^2 of the model
lm2.score(df_X, df_y)

In [ ]:
# Calculate the predicted price using the new model 
y_pred = lm2.predict(df_X)

In [ ]:
# Estimate the RMSE
root_mean_squared_error(df_y, y_pred)

In [ ]:
carat_grid = np.linspace(df['carat'].min(), df['carat'].max(), 200).reshape(-1, 1)
price_grid_pred = lm2.predict(pd.DataFrame(carat_grid, columns=['carat']))

fig, ax = plt.subplots()
ax.scatter(df['carat'], df['price'], alpha=0.2, label="Actual data")
ax.plot(carat_grid, price_grid_pred, color="red", label="Fitted model")
ax.set_xlabel("Carat")
ax.set_ylabel("Price")
ax.set_title("Linear model after removing large diamonds")
ax.legend()
plt.show()

### Adding another variable 'cut'

In [ ]:
# Let's only keep carat and cut as independent variables. 
# You can include more variables (e.g., color, clarity) later if you want. 
df_X = df[['carat','cut']]
df_y = df['price']

In [ ]:
df_X.head()

In [ ]:
# 'cut' is a categorical variable with five possible values 
df_X.cut.value_counts()

In [ ]:
# You could use sklearn's encoder such as OneHotEncoder to encode this variable. 
# Alternatively, use get_dummies() to convert 'cut' to dummy variables, i.e., one-hot encoding 
df_X_onehot = pd.get_dummies(df_X)
df_X_onehot.head()

In [ ]:
# To differentiate the five levels of cut, we only need to keep four dummy variables
df_X1 = df_X_onehot[['carat','cut_Ideal','cut_Premium','cut_Very Good','cut_Good']]

In [ ]:
# Build a new model
lm_onehot = LinearRegression()
lm_onehot.fit(df_X1, df_y)

In [ ]:
# Print the intercept and coefficients
print(lm_onehot.intercept_, lm_onehot.coef_)

**Question: How to interpret these coefficients?**

In [ ]:
# R^2 
lm_onehot.score(df_X1, df_y)

In [ ]:
# RMSE 
y_pred = lm_onehot.predict(df_X1)

root_mean_squared_error(df_y, y_pred)

### Encode 'cut' with an ordinal encoder
With the knowledge that the five values of cut, ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal'], indicate the quality from low to high, we could also encode this variable using an ordinal encoder (0, 1, 2, 3, 4).

In [ ]:
# select the variable to be encoded
cat_df = df[['cut']]
cat_df.head()

In [ ]:
# OrdinalEncoder 
from sklearn.preprocessing import OrdinalEncoder
# You can specify the order of values for encoding 
ord_encoder = OrdinalEncoder(categories=[['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']])
cat_ord_encoded = ord_encoder.fit_transform(cat_df)
ord_encoder.categories_

In [ ]:
cat_ord_encoded

In [ ]:
# convert cat_ord_encoded into dataframe
cat_df_ord_encoded = pd.DataFrame(cat_ord_encoded, columns=cat_df.columns) 
cat_df_ord_encoded

In [ ]:
cat_df.cut.value_counts()

In [ ]:
cat_df_ord_encoded.cut.value_counts()

In [ ]:
df_X_ord = pd.concat([df_X[['carat']], cat_df_ord_encoded], axis=1)
df_X_ord

In [ ]:
# Build a new model
lm_ord = LinearRegression()
lm_ord.fit(df_X_ord, df_y)

In [ ]:
# Print the intercept and coefficients
print(lm_ord.intercept_, lm_ord.coef_)

**Question: How to interpret the coefficients?**

In [ ]:
# R^2
lm_ord.score(df_X_ord, df_y)

In [ ]:
# RMSE
df_y_pred = lm_ord.predict(df_X_ord)

root_mean_squared_error(df_y, df_y_pred)

**Note:** Ordinal encoding assumes that the distance between adjacent quality levels is equal. For example, it treats the difference between Fair and Good the same as the difference between Premium and Ideal. That assumption may or may not be appropriate, so ordinal encoding should be used thoughtfully.

### Non-linearity

The relationship between carat and price seems to be non-linear. 

In [ ]:
fig, ax = plt.subplots()
sns.scatterplot(data=df, x='carat', y='price', alpha=0.5, ax=ax)
ax.set_xlabel("Carat")
ax.set_ylabel("Price")
ax.set_title("Nonlinear relationship between carat and price")
plt.show()

In [ ]:
df['log_price'] = np.log(df.price)
df['log_price']

In [ ]:
df['log_carat'] = np.log(df.carat)

In [ ]:
fig, ax = plt.subplots()
sns.scatterplot(data=df, x='log_carat', y='log_price', alpha=0.5, ax=ax)
ax.set_xlabel("log(carat)")
ax.set_ylabel("log(price)")
ax.set_title("Log-transformed carat and price")
plt.show()

In [ ]:
df_X_log = df[['log_carat']]
df_X_log

In [ ]:
df_y_log = df['log_price']
df_y_log

In [ ]:
lm_log = LinearRegression()
lm_log.fit(df_X_log, df_y_log)

In [ ]:
# Print the intercept and coefficients
print(lm_log.intercept_, lm_log.coef_)

**How to interpret the coefficients?**

In [ ]:
# R^2
lm_log.score(df_X_log, df_y_log)

In [ ]:
# Make predictions using the new model 
# using log_carat to predict log_price
log_y_pred = lm_log.predict(df_X_log)
# You must convert log price back to price 
y_pred = np.exp(log_y_pred)

In [ ]:
# RMSE 
root_mean_squared_error(df_y, y_pred)

In [ ]:
log_carat_grid = np.linspace(df['log_carat'].min(), df['log_carat'].max(), 200).reshape(-1, 1)
log_price_grid_pred = lm_log.predict(pd.DataFrame(log_carat_grid, columns=['log_carat']))

fig, ax = plt.subplots()
sns.scatterplot(data=df, x='log_carat', y='log_price', alpha=0.2, label="Actual data", ax=ax)
ax.plot(log_carat_grid, log_price_grid_pred, color="red", label="Fitted model")
ax.set_xlabel("log(carat)")
ax.set_ylabel("log(price)")
ax.set_title("Linear model on the log-transformed scale")
ax.legend()
plt.show()

In [ ]:
carat_grid = np.linspace(df['carat'].min(), df['carat'].max(), 200).reshape(-1, 1)
log_carat_grid = np.log(carat_grid)
price_grid_pred = np.exp(lm_log.predict(pd.DataFrame(log_carat_grid, columns=['log_carat'])))

fig, ax = plt.subplots()
sns.scatterplot(data=df, x='carat', y='price', alpha=0.2, label="Actual data", ax=ax)
ax.plot(carat_grid, price_grid_pred, color="red", label="Log-transformed model")
ax.set_xlabel("Carat")
ax.set_ylabel("Price")
ax.set_title("Log-transformed model on the original price scale")
ax.legend()
plt.show()

# Assignment

Build another regression model (e.g., including other variables such as color and clarity, adding categorical variable(s) to the log-transformed model). Test its performance. 